# Session 2 · Day 1 - Optional Google Earth Engine Raster to a PSGC Table

This notebook is optional enrichment. Google Earth
Engine (GEE) is a nice-to-have for the TESDA use cases, and none of the core Session 2 labs depend
on it. Skip it freely when time is short or authentication is not available. The point is to show the full
arc: a **satellite raster grid** becomes a table joined to **PSGC administrative areas** and saved
to a file for analysis anywhere.

### What this notebook covers
1. Understand **why** nightlights and population rasters help, and the grid → polygon → tabular →
   file arc
2. Read the **real GEE setup steps** (project, API, registration, common auth error)
3. Authenticate and initialize Earth Engine behind a **safe flag** (`GEE_OK`)
4. Load the **VIIRS nightlights** and **WorldPop population** rasters over the Philippines
5. Load **admin polygons** (FAO GAUL as a PSGC stand-in) and see how to swap in real PSGC boundaries
6. Run **zonal statistics** to reduce the raster grid onto each polygon
7. Turn the polygons into a **pandas table** (one row for each area)
8. **Save** the table to a file *(with an exercise)*

> **Runs without Earth Engine.** If GEE is missing or not authenticated, the notebook switches to a
> small synthetic for each province table so every later step still runs and still writes a file.

> **How to use this notebook**
> - ▶ **Run each cell in order** (`Shift+Enter`). Later cells depend on earlier ones.
> - ✏️ **Exercise cells** have `### START CODE HERE ###` … `### END CODE HERE ###`. Write
>   code between those lines only.
> - ✅ After an exercise, run the **check cell**. Fix the code until it prints a pass.
> - 🎛️ **Play-around cells** are open to tweak, change a value, re-run, see what happens.
>
> _Look for **`# >>> SWAP IN REAL DATA`** to plug in the official PSGC boundaries later._

### New to Python or pandas?

Anyone coming from another language can run `S2_commands_primer` first. It lays out the mental model in one page: a `DataFrame` is a table (a SQL table, an R `data.frame`, a spreadsheet), a `merge` is a `JOIN`, `NaN` is the missing value, and a chain of `.method().method()` reads left to right. The comments in the code cells point out each pandas idiom as it appears.

In [ ]:
# Run once per Colab session (skip if already installed locally).
!pip -q install earthengine-api geemap geopandas pandas

In [ ]:
import os
import tempfile
import numpy as np
import pandas as pd
pd.set_option("display.max_columns", 20)

## Two raster proxies and the grid-to-table arc

Two public satellite rasters are useful demand and activity proxies for TESDA planning:

- **Nightlights (VIIRS).** Collection `NOAA/VIIRS/DNB/MONTHLY_V1/VCMSLCFG`, band `avg_rad`. Average
  night-time radiance is a rough proxy for economic activity and settlement intensity.
- **Population (WorldPop).** Collection `WorldPop/GP/100m/pop`. A gridded estimate of how many
  people live in each ~100 m cell.

Both arrive as a **raster grid**: millions of small cells, each with a value. Planning happens at
the level of **administrative areas** (regions, provinces, cities), keyed by the **PSGC** code. So
the arc of this notebook is:

**raster grid → admin polygon (zonal statistics) → tabular table → saved file.**

Zonal statistics is the bridge. For each polygon a reducer summarizes the grid cells that fall inside it
(sum the population, average the nightlights), collapsing a raster into one row for each area.

### What each raster proxy captures

The two rasters measure different things, so they call for different readings. WorldPop is a modelled head count: each cell estimates how many people live there, and summing the cells over an area gives a population total suitable as a demand denominator. Nightlights measure average radiance, an intensity that tracks economic activity and settlement brightness. That radiance does not count residents. A bright commercial district can hold few people, and a dense rural barangay can stay dark. WorldPop answers how many people. Nightlights answer how much activity. Trouble starts when one is read as a substitute for the other.

## Earth Engine setup (read this first)

Earth Engine needs a one-time account and project setup. Most first-time trouble is authentication,
so follow these steps carefully.

1. **Create a clean Cloud project.** Go to
   <https://console.cloud.google.com/projectcreate> and make a new project. A fresh project avoids
   the OAuth clash described below.
2. **Enable the Earth Engine API.** Open
   <https://console.cloud.google.com/apis/library>, search for **"Earth Engine API"**, and click
   **Enable** for the project.
3. **Register the project for Earth Engine.** Go to
   <https://signup.earthengine.google.com/> and register the same project. For TESDA / DepEd use,
   choose the **noncommercial** (research / education) plan.
4. **Authenticate and initialize** in the notebook:
   ```python
   import ee
   ee.Authenticate(auth_mode='notebook')   # opens a browser link, paste the token back
   ee.Initialize(project='gee-project-id')
   ```

**Troubleshooting the exact error** `Project has an incompatible OAuth2 Client configuration`:
a Cloud project can hold only **one** OAuth2 client. When the project already has one (check under
**Cloud Console → APIs & Services → Credentials → OAuth 2.0 Client IDs**), authentication clashes.
The fix is to use a **different or brand-new project** with no existing OAuth2 client. Full guide:
<https://developers.google.com/earth-engine/guides/auth>.

## Authenticate and initialize safely

Set `GEE_PROJECT` to the target project id. The initialization here tries to start Earth Engine and sets
a module flag **`GEE_OK`**. Every GEE-dependent step later checks that flag, so the notebook keeps
running even when Earth Engine is unavailable (for example on a headless machine with no auth).

Only `ee.Initialize` runs here. Run `ee.Authenticate(auth_mode='notebook')` once, in a
separate cell, the first time a machine is set up.

In [ ]:
GEE_PROJECT = "gee-project-id"   # 👈 set to the Earth Engine-enabled Cloud project id
GEE_OK = False

try:
    import ee
    # First-time setup only (uncomment and run once, follow the browser prompt):
    # ee.Authenticate(auth_mode='notebook')
    ee.Initialize(project=GEE_PROJECT)
    GEE_OK = True
    print("Earth Engine ready. project:", GEE_PROJECT)
except Exception as e:
    print("Earth Engine not available (", type(e).__name__, ") - using the offline fallback.")
    print("  To run live: install earthengine-api, run ee.Authenticate(auth_mode='notebook'),")
    print("  set GEE_PROJECT to the project id, then re-run this cell.")

print("GEE_OK =", GEE_OK)

**Expected output (offline):** a line saying Earth Engine is not available, the setup hint, and
`GEE_OK = False`. With a live, authenticated project it shows `Earth Engine ready` and
`GEE_OK = True` instead.

## Load the rasters (the grid)

Load both ImageCollections, filter them to a recent window and to a Philippines bounding box, and
print the band names. VIIRS is monthly, so the 2023 months average into one image. WorldPop is
per-year, so the 2020 Philippines layer applies. The rasters load only when `GEE_OK` is true.

In [ ]:
PH_BBOX = ee.Geometry.Rectangle([116.9, 4.6, 126.6, 19.6]) if GEE_OK else None

if GEE_OK:
    # Nightlights: average the 2023 monthly composites, keep the avg_rad band.
    nightlights = (ee.ImageCollection("NOAA/VIIRS/DNB/MONTHLY_V1/VCMSLCFG")
                   .filterDate("2023-01-01", "2023-12-31")
                   .select("avg_rad")
                   .mean()
                   .clip(PH_BBOX))

    # Population: WorldPop 2020, Philippines, mosaicked into one image.
    worldpop = (ee.ImageCollection("WorldPop/GP/100m/pop")
                .filter(ee.Filter.eq("year", 2020))
                .filter(ee.Filter.eq("country", "PHL"))
                .mosaic()
                .clip(PH_BBOX))

    print("nightlights bands:", nightlights.bandNames().getInfo())
    print("population bands: ", worldpop.bandNames().getInfo())
else:
    print("Skipped (GEE_OK is False). The offline fallback in Step 7 builds a synthetic table.")

## Admin polygons (PSGC stand-in)

Zonal statistics needs polygons to reduce the raster onto. The official **PSGC** boundaries are the real target,
but for a runnable demo **FAO GAUL** (`FAO/GAUL/2015/level1`, first-level administrative
areas) filtered to the Philippines serves as a stand-in. Level 1 is region-like, and `level2` is
province/city-like. The `# >>> SWAP IN REAL DATA` note shows how to bring in the real PSGC file.

In [ ]:
if GEE_OK:
    # FAO GAUL level 1 as a STAND-IN for official PSGC boundaries.
    polygons = (ee.FeatureCollection("FAO/GAUL/2015/level1")
                .filter(ee.Filter.eq("ADM0_NAME", "Philippines")))
    print("admin polygons loaded:", polygons.size().getInfo())

    # >>> SWAP IN REAL DATA: replace GAUL with official PSGC boundaries.
    #   Option A - upload the PSGC shapefile as an Earth Engine asset, then:
    #       polygons = ee.FeatureCollection("projects/<project-id>/assets/psgc_regions")
    #   Option B - read a local .shp with geopandas and convert with geemap:
    #       import geopandas as gpd, geemap
    #       gdf = gpd.read_file("psgc_regions.shp").to_crs("EPSG:4326")
    #       polygons = geemap.geopandas_to_ee(gdf)
else:
    print("Skipped (GEE_OK is False).")

## Zonal statistics (grid → polygon)

`reduceRegions` runs a **reducer** over every raster cell inside each polygon. The reducer and the
scale are the two choices that matter:

- **Population uses `sum`**: people counts add up over the area, at **scale 100 m** (WorldPop's
  native cell size).
- **Nightlights uses `mean`**: radiance is an intensity, so an average is the sensible summary, 
  at **scale ~500 m** (close to VIIRS resolution). A finer scale is more precise but slower.

One reduction runs per raster, and the next step combines them into a table.

### reduceRegions vs sampleRegions

Both functions pull raster values into a FeatureCollection, but they answer different questions. `reduceRegions` aggregates every cell inside each polygon into one summary per area, a sum or a mean, which is what a per-area table needs. `sampleRegions` returns one row per pixel instead, the raw cell values under points or polygons, with no aggregation. `reduceRegions` fits when the unit of analysis is the area, and `sampleRegions` fits when the individual cell values are needed, for example to build training samples for a model.

In [ ]:
if GEE_OK:
    # Population summed per polygon.
    pop_fc = worldpop.reduceRegions(
        collection=polygons, reducer=ee.Reducer.sum(), scale=100)
    # Nightlights averaged per polygon.
    lit_fc = nightlights.reduceRegions(
        collection=polygons, reducer=ee.Reducer.mean(), scale=500)
    print("reduced population features:", pop_fc.size().getInfo())
    print("reduced nightlight features:", lit_fc.size().getInfo())
else:
    print("Skipped (GEE_OK is False).")

## Polygons to a table

With a small number of areas, `.getInfo()` pulls the FeatureCollection properties straight into pandas. Each feature becomes one row: `area_name`, `population`, `avg_nightlights`. For a
large collection, `.getInfo()` will time out, so the scalable path is `ee.batch.Export.table.toDrive`
to write a CSV to Google Drive and read it back later.

**When `GEE_OK` is false**, the cell instead builds a small **synthetic** for each province table so the
rest of the notebook still runs end to end. This fallback is clearly marked.

### getInfo vs Export.table.toDrive

Both move a result out of Earth Engine, and the right choice depends on the number of rows expected. `getInfo()` pulls the result straight into Python in the current session. It is convenient for a handful of areas, but it runs under a request timeout and fails once the collection grows large. `ee.batch.Export.table.toDrive` runs as a background task that writes a CSV to Google Drive for reading back afterwards, and it handles thousands of features without timing out. Use `getInfo()` for a small result needed right away, and the export for anything large.

In [ ]:
if GEE_OK:
    # Live path: pull the reduced properties and align on the GAUL area name.
    pop_props = {f["properties"]["ADM1_NAME"]: f["properties"].get("sum")
                 for f in pop_fc.getInfo()["features"]}
    lit_props = {f["properties"]["ADM1_NAME"]: f["properties"].get("mean")
                 for f in lit_fc.getInfo()["features"]}
    names = list(pop_props)
    zonal_df = pd.DataFrame({
        "area_name":       names,
        "population":      [pop_props[k] for k in names],
        "avg_nightlights": [lit_props.get(k) for k in names],
    })
    # For large collections use Export.table.toDrive instead of getInfo():
    #   task = ee.batch.Export.table.toDrive(collection=pop_fc, description="pop_by_area",
    #                                         fileFormat="CSV")
    #   task.start()
    print("Built the table from live Earth Engine zonal statistics.")
else:
    # ---------------- OFFLINE FALLBACK (synthetic) ----------------
    # >>> SWAP IN REAL DATA: this stands in for the live GEE zonal result above.
    rng = np.random.default_rng(42)
    provinces = ["Metro Manila", "Cebu", "Davao del Sur", "Pampanga", "Laguna",
                 "Iloilo", "Negros Occidental", "Batangas", "Cavite", "Bulacan"]
    zonal_df = pd.DataFrame({
        "area_name":       provinces,
        "population":      rng.integers(300_000, 5_000_000, size=len(provinces)),
        "avg_nightlights": rng.uniform(2.0, 60.0, size=len(provinces)).round(2),
    })
    print("GEE_OK is False -> built a SYNTHETIC per-province table (offline fallback).")

print("table shape:", zonal_df.shape)
zonal_df.head()

**Expected output (offline):** `GEE_OK is False -> built a SYNTHETIC for each province table ...`, then
`table shape: (10, 3)` and the first rows with `area_name`, `population`, and `avg_nightlights`.

## Save the table to a file

The table is now analysis-ready. It is written three ways into a temporary folder, and each
path is printed: a plain **CSV** of the attributes, a **GeoJSON** that keeps one point geometry per area,
and an **ESRI Shapefile**. The Shapefile format truncates field names to 10 characters, so
`avg_nightlights` is shortened before that write. Each format has a place: CSV for quick tabular
use, GeoJSON for web maps and readable geometry, Shapefile for older GIS tools.

In [ ]:
import geopandas as gpd
from shapely.geometry import Point

OUT_DIR = tempfile.mkdtemp(prefix="s2d1_05_")

# The zonal table has no geometry, so a representative point is attached per area.
# A live GEE run could instead keep each polygon boundary from the FeatureCollection.
pt_rng = np.random.default_rng(0)
lon = pt_rng.uniform(116.9, 126.6, size=len(zonal_df))
lat = pt_rng.uniform(4.6, 19.6, size=len(zonal_df))
gdf = gpd.GeoDataFrame(zonal_df.copy(),
                       geometry=[Point(x, y) for x, y in zip(lon, lat)],
                       crs="EPSG:4326")

# 1. CSV: attributes only.
csv_path = os.path.join(OUT_DIR, "nightlights_population_by_area.csv")
zonal_df.to_csv(csv_path, index=False)

# 2. GeoJSON: keeps the geometry, long field names are fine.
geojson_path = os.path.join(OUT_DIR, "by_area.geojson")
gdf.to_file(geojson_path, driver="GeoJSON")

# 3. Shapefile: field names are truncated to 10 characters, so rename first.
shp_path = os.path.join(OUT_DIR, "by_area.shp")
gdf.rename(columns={"avg_nightlights": "avg_nl"}).to_file(shp_path, driver="ESRI Shapefile")

print("saved CSV to:     ", csv_path)
print("saved GeoJSON to: ", geojson_path)
print("saved Shapefile to:", shp_path)
assert os.path.exists(csv_path), "CSV not written"
assert os.path.exists(geojson_path), "GeoJSON not written"
assert os.path.exists(shp_path), "Shapefile not written"
gdf.head()

## ✏️ Exercise - rank and save the top areas

Work from `zonal_df` (real or synthetic) to build a ranked table and save the top few areas. This
is verifiable offline, so it passes on the synthetic fallback too. Complete the code between the
markers so that `ranked`:

1. is a **copy** of `zonal_df` with a new column `pop_per_nightlight = population / avg_nightlights`,
2. is **sorted by `pop_per_nightlight`, highest first**,
3. has its **top `TOP_N` rows written** to `top_path` (with `index=False`).

In [ ]:
TOP_N = 5
top_path = os.path.join(OUT_DIR, "top_areas.csv")

### START CODE HERE ###   (about 4 lines)
ranked = None            # copy zonal_df, add column pop_per_nightlight = population / avg_nightlights
                         # then sort by pop_per_nightlight, highest first
                         # then write ranked.head(TOP_N) to top_path with index=False
### END CODE HERE ###

ranked.head(TOP_N) if ranked is not None else "fill in the code above"

## ✅ Check the result
Run this, it verifies the column, the sort order, and the saved file.

In [ ]:
try:
    assert ranked is not None, "ranked is None, did you fill in the code?"
    assert "pop_per_nightlight" in ranked.columns, "missing column 'pop_per_nightlight'"
    vals = ranked["pop_per_nightlight"].values
    assert np.all(np.diff(vals) <= 1e-9), "not sorted highest-first by pop_per_nightlight"
    assert os.path.exists(top_path), "top_areas.csv was not written"
    reread = pd.read_csv(top_path)
    n_expected = min(TOP_N, len(zonal_df))
    assert reread.shape == (n_expected, 4), f"expected saved shape {(n_expected, 4)}, got {reread.shape}"
    assert "pop_per_nightlight" in reread.columns, "saved CSV is missing the new column"
    print(f"✅ Passed! Wrote the top {n_expected} areas (4 columns) to", os.path.basename(top_path))
except AssertionError as e:
    print("❌ Not yet:", e)

### ❓ Question

The saved `top_areas.csv` is ranked by `pop_per_nightlight`, highest first. Which area sits at
the top, the one with the highest population per unit of nightlight? Put its `area_name` (a
string) in `q_answer`.

In [ ]:
q_answer = None   # <- answer here (the area_name string at the top of the ranking)

In [ ]:
try:
    top_area = ranked.sort_values("pop_per_nightlight", ascending=False).iloc[0]["area_name"]
    assert q_answer == top_area, f"expected {top_area!r}, got {q_answer!r}"
    print("✅ Correct! Highest population per nightlight:", top_area)
except AssertionError as e:
    print("❌ Not yet:", e)

## 🎛️ Play around - switch the raster, the reducer, or the admin level

Three modelling choices drive this pipeline: **which raster** to reduce, **which reducer** to apply,
and **which admin level** the polygons are. The offline table is summarized under the
chosen settings. With a live `GEE_OK` session, the raster-load, polygon, and zonal-statistics cells re-run at the chosen level.

In [ ]:
raster      = "population"   # 👈 try "nightlights"
reducer_name = "sum"         # 👈 try "mean"
admin_level  = 1             # 👈 try 2 for province/city polygons (needs a live GEE session)

col = "population" if raster == "population" else "avg_nightlights"
value = zonal_df[col].mean() if reducer_name == "mean" else zonal_df[col].sum()
print(f"{reducer_name} of {col} across {len(zonal_df)} areas = {value:,.2f}")
if GEE_OK:
    print(f"Live: re-run Steps 4-6 with FAO/GAUL/2015/level{admin_level} and Reducer.{reducer_name}().")

## Recap
- Satellite rasters (nightlights, population) are optional **enrichment** proxies for TESDA planning.
- The pipeline is one arc: **raster grid → admin polygon → tabular table → saved file.**
- **Zonal statistics** (`reduceRegions`) is the bridge, pick the reducer (`sum` vs `mean`) and the
  scale to match each raster.
- Guard every live call with `GEE_OK` so the notebook still finishes when Earth Engine is absent.
- The tabular and save side works the same on real or synthetic data.

**Optional next steps:** swap the GAUL stand-in for real PSGC boundaries, and join this table back
to the center accessibility work from `S2_D2_02`.

### ❓ Question (exercise)
Switch the raster from population to nightlights and re-rank the areas. Report which area ranks highest and whether the ranking differs from the population ranking.

Left as an exercise: no answer key is provided. Work it out in the cell below.

In [ ]:
# Left as an exercise. Compute and inspect the answer here.


### ❓ Question (exercise)
Change the zonal reducer from mean to sum for nightlights. Report how the top area changes.

Left as an exercise: no answer key is provided. Work it out in the cell below.

In [ ]:
# Left as an exercise. Compute and inspect the answer here.


## References & further reading

- [Earth Engine, Python installation & guide](https://developers.google.com/earth-engine/guides/python_install)
- [Earth Engine, authentication guide](https://developers.google.com/earth-engine/guides/auth)
- [Earth Engine data catalog](https://developers.google.com/earth-engine/datasets/)
- [VIIRS DNB monthly (nightlights) dataset](https://developers.google.com/earth-engine/datasets/catalog/NOAA_VIIRS_DNB_MONTHLY_V1_VCMSLCFG)
- [WorldPop 100 m population dataset](https://developers.google.com/earth-engine/datasets/catalog/WorldPop_GP_100m_pop)
- [Image.reduceRegions, reference](https://developers.google.com/earth-engine/apidocs/ee-image-reduceregions)
- [FAO GAUL 2015 administrative boundaries](https://developers.google.com/earth-engine/datasets/catalog/FAO_GAUL_2015_level1)
- [geemap, Earth Engine + Python mapping](https://geemap.org/)
- [PSA, Philippine Standard Geographic Code (PSGC)](https://psa.gov.ph/classification/psgc)